In [65]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
!pip install torch_geometric

In [67]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from torch_geometric.transforms import RandomLinkSplit

In [68]:
checkpoint = torch.load("/content/drive/MyDrive/drug_repurposing/data/processed/filtered_graph.pt", weights_only=False)

data = checkpoint["data"]
node_maps = checkpoint["node_maps"]
reverse_node_maps = checkpoint["reverse_node_maps"]

print(data)

HeteroData(
  Compound={ num_nodes=1441 },
  Disease={ num_nodes=134 },
  Gene={ num_nodes=18270 },
  (Compound, CtD, Disease)={ edge_index=[2, 487] },
  (Gene, GiG, Gene)={ edge_index=[2, 147164] },
  (Disease, DdG, Gene)={ edge_index=[2, 7623] },
  (Compound, CbG, Gene)={ edge_index=[2, 11571] },
  (Compound, CuG, Gene)={ edge_index=[2, 18756] },
  (Disease, DaG, Gene)={ edge_index=[2, 12623] },
  (Gene, GcG, Gene)={ edge_index=[2, 61690] },
  (Gene, Gr>G, Gene)={ edge_index=[2, 265672] },
  (Compound, CdG, Gene)={ edge_index=[2, 21102] },
  (Disease, DuG, Gene)={ edge_index=[2, 7731] }
)


In [69]:
torch.manual_seed(42)

In [70]:
split = RandomLinkSplit(
    num_val=0.15,
    num_test=0.15,
    is_undirected=False,
    add_negative_train_samples=False,
    edge_types=('Compound', 'CtD', 'Disease'),
)

train_data, val_data, test_data = split(data)

In [71]:
edge_store = test_data['Compound', 'CtD', 'Disease']

test_edges = edge_store.edge_label_index
test_labels = edge_store.edge_label

# keep only positive edges
pos_mask = test_labels == 1
test_edges = test_edges[:, pos_mask]

print("Positive test edges:", test_edges.shape)

Positive test edges: torch.Size([2, 73])


In [72]:
gat_artifact = torch.load("/content/drive/MyDrive/drug_repurposing/models/hetero_gat_artifact.pt", map_location="cpu", weights_only=False)

print(gat_artifact.keys())

dict_keys(['model_name', 'embedding_dim', 'heads', 'dropout_layer1', 'dropout_layer2', 'weight_decay', 'best_val_auc', 'best_val_ap', 'test_auc', 'test_ap', 'train_losses', 'val_aucs', 'val_aps', 'relation_attention', 'embeddings'])


In [73]:
z_dict = gat_artifact["embeddings"]

comp_emb = z_dict["Compound"]   # [num_compounds, dim]
dis_emb = z_dict["Disease"]     # [num_diseases, dim]

print(comp_emb.shape, dis_emb.shape)

torch.Size([1441, 32]) torch.Size([134, 32])


In [74]:
def ranking_evaluation(comp_emb, dis_emb, test_edges):

    ranks = []

    for i in tqdm(range(test_edges.shape[1])):

        drug = test_edges[0, i]
        disease = test_edges[1, i]

        disease_vec = dis_emb[disease]

        # score all compounds for this disease
        scores = torch.matmul(comp_emb, disease_vec)
        scores = scores.detach().numpy()

        # rank descending
        sorted_idx = np.argsort(-scores)

        # find rank of true drug
        rank = np.where(sorted_idx == drug.item())[0][0] + 1

        ranks.append(rank)

    ranks = np.array(ranks)

    mrr = np.mean(1.0 / ranks)
    hits1 = np.mean(ranks <= 1)
    hits5 = np.mean(ranks <= 5)
    hits10 = np.mean(ranks <= 10)

    precision10 = hits10 / 10

    return {
        "MRR": mrr,
        "Hits@1": hits1,
        "Hits@5": hits5,
        "Hits@10": hits10,
        "Precision@10": precision10
    }

In [75]:
gat_results = ranking_evaluation(comp_emb, dis_emb, test_edges)

gat_results

100%|██████████| 73/73 [00:00<00:00, 5055.88it/s]


{'MRR': np.float64(0.11120157156115408),
 'Hits@1': np.float64(0.0547945205479452),
 'Hits@5': np.float64(0.136986301369863),
 'Hits@10': np.float64(0.1780821917808219),
 'Precision@10': np.float64(0.01780821917808219)}

In [76]:
results_df = pd.DataFrame([gat_results], index=["HeteroGAT"])

results_df

,MRR,Hits@1,Hits@5,Hits@10,Precision@10
HeteroGAT,0.111202,0.054795,0.136986,0.178082,0.017808


In [77]:
results_df.to_csv("/content/drive/MyDrive/drug_repurposing/results/heterogat_ranking_results.csv", index=True)

In [78]:
print(comp_emb.shape[0])

1441


In [79]:
def get_disease_index(disease_name):
    return node_maps["Disease"].get(disease_name, None)

In [80]:
def extract_doid(disease):
    return disease.split("::")[-1]

In [81]:
def get_top_k_drugs(disease_idx, comp_emb, dis_emb, k=20):

    disease_vec = dis_emb[disease_idx]

    scores = torch.matmul(comp_emb, disease_vec)
    scores = scores.detach().numpy()

    top_k_idx = np.argsort(-scores)[:k]

    return top_k_idx, scores[top_k_idx]

In [82]:
def idx_to_drug_names(indices):
    names = []
    for i in indices:
        raw = reverse_node_maps["Compound"][int(i)]
        clean = raw.split("::")[-1]   # DB ID
        names.append(clean)
    return names

In [83]:
list(node_maps["Disease"].keys())[:20]

['Disease::DOID:0050156',
 'Disease::DOID:0050425',
 'Disease::DOID:0050741',
 'Disease::DOID:0050742',
 'Disease::DOID:0060073',
 'Disease::DOID:0060119',
 'Disease::DOID:10021',
 'Disease::DOID:1024',
 'Disease::DOID:10283',
 'Disease::DOID:10534',
 'Disease::DOID:10608',
 'Disease::DOID:10652',
 'Disease::DOID:10763',
 'Disease::DOID:10811',
 'Disease::DOID:10871',
 'Disease::DOID:1094',
 'Disease::DOID:10941',
 'Disease::DOID:10976',
 'Disease::DOID:11054',
 'Disease::DOID:11119']

In [84]:
disease_name = "Disease::DOID:0050156" #diabetes mellitus

disease_idx = get_disease_index(disease_name)

print("Disease index:", disease_idx)

Disease index: 0


In [85]:
doid = extract_doid(disease_name)
print(doid)

DOID:0050156


In [86]:
top_idx, top_scores = get_top_k_drugs(disease_idx, comp_emb, dis_emb, k=20)

drug_names = idx_to_drug_names(top_idx)

for i, (drug, score) in enumerate(zip(drug_names, top_scores), 1):
    clean = drug.split("::")[-1]
    print(f"{i}. {clean} | score={score:.4f}")

1. DB00993 | score=7.3334
2. DB00929 | score=6.7614
3. DB01172 | score=6.6852
4. DB00169 | score=6.1661
5. DB01097 | score=6.0499
6. DB00591 | score=5.2303
7. DB00250 | score=5.2259
8. DB00177 | score=4.8293
9. DB01033 | score=4.8031
10. DB01393 | score=4.7116
11. DB00275 | score=4.6799
12. DB00635 | score=4.5978
13. DB00876 | score=4.5757
14. DB01375 | score=4.4352
15. DB00700 | score=4.3496
16. DB00853 | score=4.3109
17. DB00584 | score=4.2610
18. DB00888 | score=4.1672
19. DB01581 | score=3.9613
20. DB00091 | score=3.8707


In [87]:
artifact = torch.load("/content/drive/MyDrive/drug_repurposing/models/hetero_vgae_artifact.pt", weights_only=False)

print(artifact.keys())

dict_keys(['model_name', 'embedding_dim', 'beta', 'train_losses', 'val_aucs', 'val_aps', 'test_auc', 'test_ap', 'embeddings', 'decoder_state_dict'])


In [88]:
z_dict = artifact["embeddings"]

comp_emb = z_dict["Compound"]
dis_emb = z_dict["Disease"]

print(comp_emb.shape, dis_emb.shape)

torch.Size([1441, 128]) torch.Size([134, 128])


In [89]:
import torch.nn as nn

class MLPDecoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Linear(dim, 1)
        )

    def forward(self, z_src, z_dst):
        x = torch.cat([z_src, z_dst], dim=-1)
        return self.mlp(x).squeeze()

In [90]:
decoder = MLPDecoder(comp_emb.shape[1])
decoder.eval()

MLPDecoder(
  (mlp): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [91]:
def ranking_with_decoder(comp_emb, dis_emb, test_edges, decoder):

    ranks = []

    for i in tqdm(range(test_edges.shape[1])):

        drug = test_edges[0, i]
        disease = test_edges[1, i]

        disease_vec = dis_emb[disease].unsqueeze(0)

        # score ALL compounds
        disease_expand = disease_vec.repeat(comp_emb.shape[0], 1)

        scores = decoder(comp_emb, disease_expand)
        scores = scores.detach().numpy()

        sorted_idx = np.argsort(-scores)

        rank = np.where(sorted_idx == drug.item())[0][0] + 1

        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "MRR": np.mean(1.0 / ranks),
        "Hits@1": np.mean(ranks <= 1),
        "Hits@5": np.mean(ranks <= 5),
        "Hits@10": np.mean(ranks <= 10),
        "Precision@10": np.mean(ranks <= 10) / 10
    }

In [92]:
vgae_results = ranking_with_decoder(comp_emb, dis_emb, test_edges, decoder)

vgae_results

100%|██████████| 73/73 [00:00<00:00, 316.67it/s]


{'MRR': np.float64(0.004748132897003706),
 'Hits@1': np.float64(0.0),
 'Hits@5': np.float64(0.0),
 'Hits@10': np.float64(0.0),
 'Precision@10': np.float64(0.0)}

In [93]:
disease_name = "Disease::DOID:0050156"

disease_idx = node_maps["Disease"][disease_name]

print(disease_idx)

0


In [94]:
def get_top_k_drugs(disease_idx, comp_emb, dis_emb, k=20):

    disease_vec = dis_emb[disease_idx]

    scores = torch.matmul(comp_emb, disease_vec)
    scores = scores.detach().numpy()

    top_k = np.argsort(-scores)[:k]

    return top_k, scores[top_k]

In [95]:
top_idx, top_scores = get_top_k_drugs(disease_idx, comp_emb, dis_emb, k=20)

drug_names = [reverse_node_maps["Compound"][int(i)] for i in top_idx]

for i, (drug, score) in enumerate(zip(drug_names, top_scores), 1):
    clean = drug.split("::")[-1]
    print(f"{i}. {clean} | score={score:.4f}")

1. DB01615 | score=22.8505
2. DB00120 | score=21.9272
3. DB00746 | score=21.0695
4. DB00517 | score=20.4736
5. DB01039 | score=20.3164
6. DB00199 | score=18.5025
7. DB00871 | score=17.7661
8. DB01369 | score=17.7008
9. DB00350 | score=17.5206
10. DB02703 | score=17.1424
11. DB01062 | score=16.9313
12. DB00228 | score=16.8902
13. DB06148 | score=16.5665
14. DB00247 | score=16.5661
15. DB00940 | score=16.5648
16. DB00266 | score=16.5596
17. DB00858 | score=16.4112
18. DB00193 | score=16.3354
19. DB01226 | score=16.0896
20. DB01094 | score=16.0482
